In [21]:
import pandas as pd

#Carregar#

In [22]:
df = pd.read_csv('assets/workouts.csv')

Visão Geral

In [23]:
print('Shape', df.shape)  #linhas e colunas
print('\n---- Primeiras Linhas ----')
print(df.head())

Shape (114, 8)

---- Primeiras Linhas ----
         data         exercicio grupo_muscular  serie  repeticoes  peso_kg  \
0  2026-01-05       Supino Reto          Peito      1          10       80   
1  2026-01-05       Supino Reto          Peito      2           8       90   
2  2026-01-05       Supino Reto          Peito      3           6       95   
3  2026-01-05  Supino Inclinado          Peito      1          10       70   
4  2026-01-05  Supino Inclinado          Peito      2           8       75   

   duracao_min           observacao  
0           45     Aquecimento leve  
1           45                  NaN  
2           45  Falha na última rep  
3           45                  NaN  
4           45                  NaN  


In [24]:
print('\n -- Tipos de Dados --') # Tipos de Dados
print(df.info())


 -- Tipos de Dados --
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   data            114 non-null    object
 1   exercicio       114 non-null    object
 2   grupo_muscular  114 non-null    object
 3   serie           114 non-null    int64 
 4   repeticoes      114 non-null    int64 
 5   peso_kg         114 non-null    int64 
 6   duracao_min     114 non-null    int64 
 7   observacao      9 non-null      object
dtypes: int64(4), object(4)
memory usage: 7.3+ KB
None


3. VALORES FALTANTES 

In [25]:
print('\n --- Nulos por Coluna ---')
print(df.isnull().sum())


 --- Nulos por Coluna ---
data                0
exercicio           0
grupo_muscular      0
serie               0
repeticoes          0
peso_kg             0
duracao_min         0
observacao        105
dtype: int64


Distribuição

In [26]:
print('\n --- Exercicios únicos ---')
print(df['exercicio'].value_counts())


 --- Exercicios únicos ---
exercicio
Supino Reto          15
Agachamento          15
Crucifixo            12
Puxada Frente        12
Remada Curvada       12
Supino Inclinado      9
Extensora             9
Leg Press             9
Remada Unilateral     6
Desenvolvimento       6
Elevação Lateral      6
Flexora               3
Name: count, dtype: int64


In [27]:
print('\n Grupos Musculares')
print(df['grupo_muscular'].value_counts)


 Grupos Musculares
<bound method IndexOpsMixin.value_counts of 0       Peito
1       Peito
2       Peito
3       Peito
4       Peito
        ...  
109    Costas
110    Costas
111    Costas
112    Costas
113    Costas
Name: grupo_muscular, Length: 114, dtype: object>


In [28]:
print('\n Estatíticas Numéricas ')
print(df[['repeticoes', 'peso_kg', 'duracao_min']].describe())


 Estatíticas Numéricas 
       repeticoes     peso_kg  duracao_min
count  114.000000  114.000000   114.000000
mean    10.219298   80.421053    52.710526
std      2.038387   53.251930     6.630212
min      6.000000   12.000000    45.000000
25%      8.000000   50.500000    48.000000
50%     10.000000   72.000000    50.000000
75%     12.000000   96.500000    60.000000
max     15.000000  260.000000    65.000000


5 . Limpeza

In [29]:
df['data'] = pd.to_datetime(df['data'])
df['observacao'] = df['observacao'].fillna('')
df['exercicio'] = df['exercicio'].str.strip().str.title()

print('\n Após Limpeza ')
print(df.dtypes)


 Após Limpeza 
data              datetime64[ns]
exercicio                 object
grupo_muscular            object
serie                      int64
repeticoes                 int64
peso_kg                    int64
duracao_min                int64
observacao                object
dtype: object


#Primeira Análise

In [30]:
treinos_por_mes = df.groupby(df['data'].dt.to_period('M'))['data'].nunique()
print("\n ----Treinos por mês -----")
print(treinos_por_mes)


 ----Treinos por mês -----
data
2026-01    12
2026-02     4
Freq: M, Name: data, dtype: int64


# Desafios #

In [31]:
# Básico: Qual exercício aparece mais vezes no dataset?
print(df['exercicio'].value_counts().index[0])
print(df['exercicio'].mode())

Supino Reto
0    Agachamento
1    Supino Reto
Name: exercicio, dtype: object


In [44]:
# Qual dia da semana tem mais treinos? (use dt.day_name())
dias_unicos = df.drop_duplicates(subset='data')
print(f"Total de treinos: {len(dias_unicos)}")

Total de treinos: 16


In [45]:
dias_unicos['dia_semana'] = dias_unicos['data'].dt.day_name()
print(dias_unicos[['data', 'dia_semana']].head())

         data dia_semana
0  2026-01-05     Monday
9  2026-01-07  Wednesday
18 2026-01-09     Friday
27 2026-01-12     Monday
36 2026-01-14  Wednesday


C:\Users\User\AppData\Local\Temp\ipykernel_14632\205371681.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dias_unicos['dia_semana'] = dias_unicos['data'].dt.day_name()


In [48]:
freq_dia = dias_unicos.groupby('dia_semana')['data'].count().sort_values(ascending=False)
print(freq_dia)

dia_semana
Monday       6
Friday       5
Wednesday    5
Name: data, dtype: int64


In [46]:
dias_unicos.groupby(dias_unicos['data'].dt.day_name())['data'].count()

data
Friday       5
Monday       6
Wednesday    5
Name: data, dtype: int64

In [50]:
# Calcule o volume total por treino (soma de repeticoes × peso_kg agrupado por data).
df['volume'] = df['repeticoes'] * df['peso_kg']

In [52]:
volume_por_treino = df.groupby('data')['volume'].sum().sort_values(ascending=False)

In [55]:
print(volume_por_treino.head())

data
2026-01-07    12110
2026-01-14    11906
2026-01-30    10942
2026-01-16     5480
2026-02-06     5222
Name: volume, dtype: int64
